# E-commerce Orders Analysis

This project analyzes a multi-table e-commerce dataset using Python, MySQL, SQL, and Power BI.

The goal is to inspect and validate the raw tables, load them into MySQL, perform SQL-based joins and analysis, and build a dashboard to summarize business insights.


In [1]:
import pandas as pd
from pathlib import Path
from getpass import getpass
from sqlalchemy import create_engine

In [2]:
project_folder = Path.cwd()
raw_folder = project_folder/"raw_data"
clean_folder = project_folder/"clean_data"
output_folder = project_folder / "output"
sql_folder = project_folder / "sql"


In [3]:
def audit_df(df, name):
    print(f"Audit: {name}")
    print("Shape:", df.shape)
    print("Missing values:")
    print(df.isna().sum())
    print("Duplicate rows:", df.duplicated().sum())
    display(df.head())

In [4]:
customers = pd.read_csv(raw_folder/"df_Customers.csv")
orderitems= pd.read_csv(raw_folder/"df_OrderItems.csv")
orders = pd.read_csv(raw_folder/"df_Orders.csv")
payments = pd.read_csv(raw_folder/"df_Payments.csv")
products = pd.read_csv(raw_folder/"df_Products.csv")


In [5]:
audit_df(customers, "customer data")
audit_df(orderitems, "order item data")
audit_df(orders, "order data")
audit_df(payments, "payment data")
audit_df(products, "product data")

Audit: customer data
Shape: (89316, 4)
Missing values:
customer_id                 0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Duplicate rows: 0


,customer_id,customer_zip_code_prefix,customer_city,customer_state
0,hCT0x9JiGXBQ,58125,varzea paulista,SP
1,PxA7fv9spyhx,3112,armacao dos buzios,RJ
2,g3nXeJkGI0Qw,4119,jandira,SP
3,EOEsCQ6QlpIg,18212,uberlandia,MG
4,mVz5LO2Vd6cL,88868,ilhabela,SP


Audit: order item data
Shape: (89316, 5)
Missing values:
order_id            0
product_id          0
seller_id           0
price               0
shipping_charges    0
dtype: int64
Duplicate rows: 0


,order_id,product_id,seller_id,price,shipping_charges
0,Axfy13Hk4PIk,90K0C1fIyQUf,ZWM05J9LcBSF,223.51,84.65
1,v6px92oS8cLG,qejhpMGGVcsl,IjlpYfhUbRQs,170.80,23.79
2,Ulpf9skrhjfm,qUS5d2pEAyxJ,77p2EYxcM9MD,64.40,17.38
3,bwJVWupf2keN,639iGvMyv0De,jWzS0ayv9TGf,264.50,30.72
4,Dd0QnrMk9Cj5,1lycYGcsic2F,l1pYW6GBnPMr,779.90,30.66


Audit: order data
Shape: (89316, 7)
Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   9
order_delivered_timestamp        1889
order_estimated_delivery_date       0
dtype: int64
Duplicate rows: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_timestamp,order_estimated_delivery_date
0,Axfy13Hk4PIk,hCT0x9JiGXBQ,delivered,2017-10-22 18:57:54,2017-10-22 19:14:13,2017-10-26 22:19:52,2017-11-09
1,v6px92oS8cLG,PxA7fv9spyhx,delivered,2018-06-20 21:40:31,2018-06-20 22:20:20,2018-07-03 22:51:22,2018-07-24
2,Ulpf9skrhjfm,g3nXeJkGI0Qw,delivered,2018-02-16 16:19:31,2018-02-17 16:15:35,2018-02-27 01:29:50,2018-03-08
3,bwJVWupf2keN,EOEsCQ6QlpIg,delivered,2018-08-18 18:04:29,2018-08-18 18:15:16,2018-08-27 20:03:51,2018-09-19
4,Dd0QnrMk9Cj5,mVz5LO2Vd6cL,delivered,2017-12-22 16:44:04,2017-12-22 17:31:31,2018-01-05 19:22:49,2018-01-18


Audit: payment data
Shape: (89316, 5)
Missing values:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
Duplicate rows: 0


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,Axfy13Hk4PIk,1,credit_card,1,259.14
1,v6px92oS8cLG,1,credit_card,8,382.39
2,Ulpf9skrhjfm,1,credit_card,4,249.25
3,bwJVWupf2keN,1,credit_card,2,27.79
4,Dd0QnrMk9Cj5,1,credit_card,1,76.15


Audit: product data
Shape: (89316, 6)
Missing values:
product_id                 0
product_category_name    308
product_weight_g          15
product_length_cm         15
product_height_cm         15
product_width_cm          15
dtype: int64
Duplicate rows: 61865


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,90K0C1fIyQUf,toys,491.0,19.0,12.0,16.0
1,qejhpMGGVcsl,watches_gifts,440.0,18.0,14.0,17.0
2,qUS5d2pEAyxJ,costruction_tools_garden,2200.0,16.0,16.0,16.0
3,639iGvMyv0De,toys,1450.0,68.0,3.0,48.0
4,1lycYGcsic2F,toys,300.0,17.0,4.0,12.0


## Innitial data quality findings

- The innitial audit indicates that the **customer**, **order item** and **payment** tables contain no missing values or duplicate rows, suggesting good data completeness and no obvious duplication issues at the row level.

- These checks primarily assess completeness and duplication; further validation is required to ensure accuracy, consistency, and relational integrity.

- The **orders** table contains missing values in `order_approved_at` and `order_delivered_timestamp`. These will be further analyzed by `order_status` to determine whether the missing values are expected (e.g. cancelled or unfulfilled orders) or indicate data quality issues.

- The **products** table contains missing values in product category and dimension fields, as well as a high number of duplicate rows. The next step is to verify whether duplicate `product_id` entries have consistent attributes, and then deduplicate the dataset to construct a clean product-level table.

- Next, targeted data quality checks will be performed, including validation of key relationships and consistency across tables, before loading the cleaned datasets into MySQL for SQL analysis.


In [6]:
print("Customer ID duplicates:" , customers['customer_id'].duplicated().sum())
print("Product ID duplicates:" , products['product_id'].duplicated().sum())
print("Order ID duplicates:" , orders['order_id'].duplicated().sum())

Customer ID duplicates: 0
Product ID duplicates: 61865
Order ID duplicates: 0


In [7]:
product_consistency = products.groupby("product_id").nunique()

product_consistency.max()


product_category_name    1
product_weight_g         1
product_length_cm        1
product_height_cm        1
product_width_cm         1
dtype: int64

In [8]:
products_clean=products.drop_duplicates(subset=['product_id']).reset_index(drop=True)

In [9]:
audit_df(products_clean, "cleaned product data")

Audit: cleaned product data
Shape: (27451, 6)
Missing values:
product_id                 0
product_category_name    141
product_weight_g           2
product_length_cm          2
product_height_cm          2
product_width_cm           2
dtype: int64
Duplicate rows: 0


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,90K0C1fIyQUf,toys,491.0,19.0,12.0,16.0
1,qejhpMGGVcsl,watches_gifts,440.0,18.0,14.0,17.0
2,qUS5d2pEAyxJ,costruction_tools_garden,2200.0,16.0,16.0,16.0
3,639iGvMyv0De,toys,1450.0,68.0,3.0,48.0
4,1lycYGcsic2F,toys,300.0,17.0,4.0,12.0


## Product Table Deduplication

The products table contained repeated identical rows. After removing exact duplicates, the table was reduced from 89,316 rows to 27,451 unique product records.

After deduplication, each product record is unique and the table is more suitable to use as a product lookup table for joins.


In [10]:
missing_category_count = products_clean["product_category_name"].isna().sum()
missing_category_pct = products_clean["product_category_name"].isna().mean() * 100

print("Missing product categories:", missing_category_count)
print("Missing product category percentage:", round(missing_category_pct, 2))

Missing product categories: 141
Missing product category percentage: 0.51


In [11]:
products_clean['product_category_name']=products_clean['product_category_name'].fillna('unknown')

In [12]:
audit_df(products_clean, "cleaned product data after category handling")


Audit: cleaned product data after category handling
Shape: (27451, 6)
Missing values:
product_id               0
product_category_name    0
product_weight_g         2
product_length_cm        2
product_height_cm        2
product_width_cm         2
dtype: int64
Duplicate rows: 0


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,90K0C1fIyQUf,toys,491.0,19.0,12.0,16.0
1,qejhpMGGVcsl,watches_gifts,440.0,18.0,14.0,17.0
2,qUS5d2pEAyxJ,costruction_tools_garden,2200.0,16.0,16.0,16.0
3,639iGvMyv0De,toys,1450.0,68.0,3.0,48.0
4,1lycYGcsic2F,toys,300.0,17.0,4.0,12.0


### Missing Value Handling

The `product_category_name` field contained a small proportion of missing values (~0.51%). 

These were imputed as `"unknown"` to preserve all product records for sql analysis, particularly for categorical aggregations. Given the lack of additional information, it was not feasible to accurately infer the missing categories.

Product dimension fields (`weight`, `length`, `height`, `width`) contained only 2 missing records. These were retained as null values (`NaN`), as imputing physical measurements would introduce inaccuracies, and the negligible proportion does not materially impact analysis.

## Order Data Investigation

The orders dataset contains no duplicate rows. 

The `order_delivered_timestamp` field has 1,889 missing values, which require validation against `order_status` to determine whether they are expected (e.g. cancelled or unfulfilled orders).

The `order_approved_at` field has 9 missing values (~0.01%) and was retained as null (`NaN`) due to its negligible impact.


## Exploratory Observations

Inconsistencies were identified where cancelled orders contain delivery timestamps, violating expected business logic.

These were validated programmatically in pandas, and a flag was created to identify such records without modifying the original data.

In [13]:
orders["invalid_delivery_flag"] = (
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_timestamp"].notna())
)

In [14]:
orders[orders["invalid_delivery_flag"]]


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_timestamp,order_estimated_delivery_date,invalid_delivery_flag
14190,prmhq2USuP5I,svloFqXbk2Im,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-11-09 14:53:50,2016-12-08,True
43014,b34KqtVskWuf,fvW8GM7pZATn,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-19 18:47:43,2016-11-30,True
45856,STGODjAZ1w0n,SOg6P6UXk8Nu,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-03-21 22:03:51,2018-03-09,True
64316,gaNTtRHGtguq,Dq969FRq9Y67,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-16 14:36:59,2016-11-30,True
77637,F0WYhc9fTR0C,G0qHvWWfuMUu,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-14 15:07:11,2016-11-29,True


In [15]:
orders.groupby("order_status")["order_delivered_timestamp"].apply(lambda x: x.isna().sum())

order_status
approved         2
canceled       404
delivered        6
invoiced       266
processing     273
shipped        936
unavailable      2
Name: order_delivered_timestamp, dtype: int64

In [16]:
orders['missing_delivery_timestamp']=(
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_timestamp"].isna())
)

In [17]:
orders[orders['missing_delivery_timestamp']]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_timestamp,order_estimated_delivery_date,invalid_delivery_flag,missing_delivery_timestamp
4254,t5kBjEdw3z8t,ZeIXZ7BBF5PP,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,NaN,2017-12-18,False,True
13268,BxxG20QRNt7K,RWhfv6bIMVsQ,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,NaN,2018-07-16,False,True
30892,yRSiD6TjWuMp,WrJg7MJuozgD,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,NaN,2018-07-30,False,True
65263,1XWJAxXBtYuW,Z5AHwQM14J3A,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,NaN,2018-06-26,False,True
65525,H6OIR9EVCoos,53rYCxN5t4tp,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,NaN,2018-07-19,False,True
67317,EznMwXRSIZcm,RMR2x4OEJS2K,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,NaN,2018-07-30,False,True


## Missing delivery timestamps analyzed against `order_status`

The majority of missing values correspond to non-delivered statuses (e.g. cancelled, processing, shipped), which is expected.

However, 6 records were identified and flagged where orders marked as "delivered" lack delivery timestamps, indicating data inconsistencies.

In [18]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_timestamp",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")


In [19]:
audit_df(orders,'date standardisation of order data')

Audit: date standardisation of order data
Shape: (89316, 9)
Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   9
order_delivered_timestamp        1889
order_estimated_delivery_date       0
invalid_delivery_flag               0
missing_delivery_timestamp          0
dtype: int64
Duplicate rows: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_timestamp,order_estimated_delivery_date,invalid_delivery_flag,missing_delivery_timestamp
0,Axfy13Hk4PIk,hCT0x9JiGXBQ,delivered,2017-10-22 18:57:54,2017-10-22 19:14:13,2017-10-26 22:19:52,2017-11-09,False,False
1,v6px92oS8cLG,PxA7fv9spyhx,delivered,2018-06-20 21:40:31,2018-06-20 22:20:20,2018-07-03 22:51:22,2018-07-24,False,False
2,Ulpf9skrhjfm,g3nXeJkGI0Qw,delivered,2018-02-16 16:19:31,2018-02-17 16:15:35,2018-02-27 01:29:50,2018-03-08,False,False
3,bwJVWupf2keN,EOEsCQ6QlpIg,delivered,2018-08-18 18:04:29,2018-08-18 18:15:16,2018-08-27 20:03:51,2018-09-19,False,False
4,Dd0QnrMk9Cj5,mVz5LO2Vd6cL,delivered,2017-12-22 16:44:04,2017-12-22 17:31:31,2018-01-05 19:22:49,2018-01-18,False,False


Date columns were converted to datetime format using `pd.to_datetime()`. The missing value counts remained unchanged after conversion, indicating that no additional date values failed to parse during standardisation.

In [20]:
payments.dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [21]:
orderitems.dtypes

order_id             object
product_id           object
seller_id            object
price               float64
shipping_charges    float64
dtype: object

In [22]:
payments[["payment_value", "payment_installments"]].describe()

,payment_value,payment_installments
count,89316.000000,89316.000000
mean,268.657190,2.965717
std,344.409566,2.796406
min,0.000000,0.000000
25%,84.340000,1.000000
50%,171.860000,2.000000
75%,313.530000,4.000000
max,7274.880000,24.000000


In [23]:
orderitems[["price", "shipping_charges"]].describe()

,price,shipping_charges
count,89316.000000,89316.000000
mean,340.900543,44.283210
std,557.459897,37.672491
min,0.850000,0.000000
25%,59.650000,20.110000
50%,136.900000,35.055000
75%,399.200000,57.190000
max,6735.000000,409.680000


## Numeric Field Validation

The key numeric fields in the `payments` and `orderitems` tables were reviewed before loading into MySQL.

The fields `payment_value`, `payment_installments`, `price`, and `shipping_charges` are stored as numeric data types, making them suitable for calculations.

Summary statistics showed no negative values in these fields. Zero values were present in some payment and shipping fields, which may represent free shipping, voucher usage, or special order/payment cases. These values were retained for analysis because there is not enough evidence to treat them as invalid.


## Final Validation Before SQL Upload

The key data quality checks have been completed before loading the tables into MySQL.

- Customer, order, order item, and payment tables were checked for missing values and duplicate rows.
- Key identifiers such as `customer_id`, `order_id`, and `product_id` were reviewed for expected uniqueness.
- The products table was deduplicated to create a clean product lookup table.
- Missing product categories were labeled as `unknown`.
- Missing product dimension values were left unchanged because there was no reliable source to infer them.
- Order timestamp issues were reviewed against `order_status`, and suspicious delivery records were flagged.
- Numeric fields such as `price`, `shipping_charges`, and `payment_value` were confirmed to be numeric and suitable for calculations.

Based on these checks, the cleaned tables are ready to be loaded into MySQL for relationship checks and SQL-based analysis.


In [31]:
cleaned_tables= {
    "customers_clean":customers,
    "order_items_clean":orderitems,
    "orders_clean":orders,
    "payments_clean":payments,
    "products_clean":products
}
for table_name,df in cleaned_tables.items():
    clean_folder_path=clean_folder / f"{table_name}.csv"
    df.to_csv(clean_folder_path, index=False)
    print(f"Saved {table_name}: {len(df)} rows -> {clean_folder_path}")

Saved customers_clean: 89316 rows -> C:\Users\Ethan\Documents\ecom_analysis_project\clean_data\customers_clean.csv
Saved order_items_clean: 89316 rows -> C:\Users\Ethan\Documents\ecom_analysis_project\clean_data\order_items_clean.csv
Saved orders_clean: 89316 rows -> C:\Users\Ethan\Documents\ecom_analysis_project\clean_data\orders_clean.csv
Saved payments_clean: 89316 rows -> C:\Users\Ethan\Documents\ecom_analysis_project\clean_data\payments_clean.csv
Saved products_clean: 89316 rows -> C:\Users\Ethan\Documents\ecom_analysis_project\clean_data\products_clean.csv


The cleaned tables were exported as CSV files to the `clean_data` folder. CSV format was used because these files are intended for loading into MySQL and future analysis workflows.


## Required Packages

If needed, install the required packages with:

`%pip install pandas openpyxl sqlalchemy pymysql`


In [32]:
mysql_user = "root"
mysql_password = getpass("Enter MySQL password: ")
mysql_host = "localhost"
mysql_port = 3306
mysql_database = "ecom_analysis"

engine = create_engine(
    f"mysql+pymysql://{mysql_user}:{mysql_password}@{mysql_host}:{mysql_port}/{mysql_database}"
)

Enter MySQL password:  ········


In [33]:
sql_tables = {
    "customers": customers,
    "orders": orders,
    "order_items": orderitems,
    "payments": payments,
    "products": products_clean
}

for table_name, df in sql_tables.items():
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False
    )
    
    row_count = pd.read_sql(
        f"SELECT COUNT(*) AS total_rows FROM {table_name}",
        engine
    )
    
    print(f"Uploaded table: {table_name}")
    display(row_count)


Uploaded table: customers


,total_rows
0,89316


Uploaded table: orders


,total_rows
0,89316


Uploaded table: order_items


,total_rows
0,89316


Uploaded table: payments


,total_rows
0,89316


Uploaded table: products


,total_rows
0,27451


## MySQL Upload Validation

The MySQL row counts matched the pandas row counts after upload.

The `customers`, `orders`, `order_items`, and `payments` tables each uploaded successfully with 89,316 rows.

The cleaned `products` table uploaded successfully with 27,451 rows after removing duplicate product records.

The tables are now ready for SQL relationship checks and business analysis.
